# Lecture 3 — Class Exercise
## Line charts: CO₂ emissions

This notebook completes both chart tasks using the course file
`co2_emissions.csv` (15 countries, 2000–2022).

**Kaggle:** import this notebook and attach the course CSV as a notebook input.
The loading cell finds it under `/kaggle/input`. Run all cells to generate the
exploration outputs and both charts, then save a version and download the notebook.

**Local/GitHub:** place the notebook at `week03/lecture03_exercise.ipynb` and the
CSV at `data/co2_emissions.csv`. Python, pandas and Plotly are required.

Source stated in the exercise: [Our World in Data](https://ourworldindata.org/co2-emissions).
The supplied `CO2_per_capita` values are used unchanged; their unit is not stated
in the exercise. `CO2_Mt` represents total emissions in million tonnes.


In [9]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Optional: enter the exact CSV path if more than one matching file is attached.
DATA_PATH = None

if DATA_PATH is None:
    kaggle_input = Path('/kaggle/input')
    candidates = sorted(kaggle_input.rglob('co2_emissions.csv')) if kaggle_input.exists() else []
    if not candidates:
        candidates = [p for p in [
            Path('../data/co2_emissions.csv'),
            Path('data/co2_emissions.csv'),
            Path('co2_emissions.csv'),
        ] if p.is_file()]
    candidates = list(dict.fromkeys(p.resolve() for p in candidates))
    if not candidates:
        raise FileNotFoundError(
            'Attach the full course co2_emissions.csv to this Kaggle notebook, '
            'or place it at ../data/co2_emissions.csv. The pasted preview is not the full dataset.'
        )
    if len(candidates) > 1:
        raise ValueError(f'Multiple CSV files found. Set DATA_PATH to one of: {candidates}')
    DATA_PATH = candidates[0]

df = pd.read_csv(DATA_PATH)
required = ['Country', 'Region', 'Year', 'CO2_Mt', 'CO2_per_capita']
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

df['Country'] = df['Country'].str.strip()
df['Region'] = df['Region'].str.strip()
for column in ['Year', 'CO2_Mt', 'CO2_per_capita']:
    df[column] = pd.to_numeric(df[column], errors='raise')
if df[required].isna().any().any():
    raise ValueError('The course dataset contains missing values. Inspect the CSV before plotting.')
if df.duplicated(['Country', 'Year']).any():
    raise ValueError('Duplicate country/year rows found. Inspect the CSV before plotting.')
if not df['Year'].mod(1).eq(0).all():
    raise ValueError('Year must contain whole years.')
df['Year'] = df['Year'].astype(int)

expected_countries = {
    'United States', 'China', 'India', 'Germany', 'United Kingdom',
    'France', 'Brazil', 'Japan', 'Canada', 'Australia', 'South Korea',
    'Russia', 'South Africa', 'Mexico', 'Indonesia',
}
if set(df['Country']) != expected_countries:
    raise ValueError('The country list does not match the 15-country course dataset.')
expected_years = set(range(2000, 2023))
if len(df) != 345 or any(set(g['Year']) != expected_years for _, g in df.groupby('Country')):
    raise ValueError('Expected 345 rows with one observation per country per year from 2000 to 2022.')
if df.groupby('Country')['Region'].nunique().ne(1).any():
    raise ValueError('Each country must have a consistent Region classification.')
if (df[['CO2_Mt', 'CO2_per_capita']] < 0).any().any():
    raise ValueError('Negative emissions found. Inspect the CSV before plotting.')

print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | "
      f"Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [10]:
# Explore before building.
print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(
    df.loc[df['Year'].eq(2022)]
      .groupby('Region')['CO2_Mt']
      .mean()
      .sort_values(ascending=False)
      .round(1)
)


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


### Shared chart styling

Draw the grey lines first so the coloured line remains visible on top.
Calculate each insight title from the highlighted country's first and last
observations. This describes the change between the endpoints, without assuming
that emissions moved in the same direction every year.

In [11]:
def highlight_chart(data, metric, country, metric_name, y_title,
                    colour='#007F86', number_format=',.2f', unit=''):
    data = data.sort_values(['Country', 'Year'])
    selected = data.loc[data['Country'].eq(country)]
    observed = selected.dropna(subset=[metric])
    if len(observed) < 2:
        raise ValueError(f"At least two observations are needed for {country}.")

    if country not in data['Country'].unique():
        raise ValueError(f'{country} is not in the filtered dataset.')

    first, last = observed.iloc[0], observed.iloc[-1]
    start, end = float(first[metric]), float(last[metric])
    start_year, end_year = int(first['Year']), int(last['Year'])

    # Build an insight title from the data rather than guessing a trend.
    if end == start:
        story = 'ended at its starting level'
    elif start == 0:
        story = f'changed from {start:{number_format}} to {end:{number_format}}{unit}'
    else:
        change = (end / start - 1) * 100
        direction = 'rose' if change > 0 else 'fell'
        story = f'{direction} {abs(change):.1f}%'

    fig = go.Figure()
    hover = (
        '<b>%{fullData.name}</b><br>Year: %{x:.0f}<br>'
        + y_title + ': %{y:' + number_format + '}<extra></extra>'
    )

    # Context: every other country, in the exact grey requested.
    others = data.loc[data['Country'].ne(country)]
    for name, group in others.groupby('Country'):
        fig.add_trace(go.Scatter(
            x=group['Year'], y=group[metric], name=name, mode='lines',
            line=dict(color='#EEEEEE', width=1.2),
            connectgaps=False, hovertemplate=hover,
        ))

    # Add the highlighted country last, above the context lines.
    fig.add_trace(go.Scatter(
        x=selected['Year'], y=selected[metric], name=country, mode='lines',
        line=dict(color=colour, width=3),
        connectgaps=False, hovertemplate=hover,
    ))

    # A direct end label replaces the legend.
    fig.add_annotation(
        x=end_year, y=end,
        text=f'<b>{country}</b><br>{end:{number_format}}{unit}',
        showarrow=False, xanchor='left', xshift=10, align='left',
        font=dict(color=colour, size=13),
    )

    fig.update_layout(
        template='simple_white',
        title=dict(
            text=(
                f'<b>{country}: {metric_name} {story}</b><br>'
                f'<sup>{start_year}–{end_year}; '
                f"{data['Country'].nunique()} countries shown</sup>"
            ),
            x=0, xanchor='left', font=dict(size=20),
        ),
        showlegend=False,
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(family='Arial, sans-serif', size=13, color='#444444'),
        height=500, margin=dict(l=75, r=180, t=100, b=65),
        hovermode='closest',
    )
    fig.update_xaxes(
        title=None, showline=False, showgrid=False, zeroline=False,
        ticks='', tickangle=0, tickformat='d',
        tickvals=list(range(int(data['Year'].min()), int(data['Year'].max()) + 1, 5))
                 + ([int(data['Year'].max())]
                    if (int(data['Year'].max()) - int(data['Year'].min())) % 5 else []),
        range=[int(data['Year'].min()) - 0.3, int(data['Year'].max()) + 0.5],
    )
    fig.update_yaxes(
        title=y_title, showline=False, showgrid=False, zeroline=False,
        ticks='', tickformat=number_format, rangemode='tozero', nticks=6,
    )
    return fig


### Task 1 — CO₂ per capita for all countries

Highlight: **United States**. All other countries remain visible in grey.

In [12]:
fig1 = highlight_chart(
    data=df,
    metric='CO2_per_capita',
    country='United States',
    metric_name='CO₂ per capita',
    y_title='CO₂ per capita',
)
fig1.show()


### Task 2 — Total CO₂ emissions for Asian countries

Highlight: **China**. Use the CSV's `Region` classification to include every
Asian country in this dataset. Plot total emissions (`CO2_Mt`) for this task.

In [13]:
asia = df.loc[df['Region'].eq('Asia')].copy()
print("Asian countries:", ', '.join(sorted(asia['Country'].unique())))

fig2 = highlight_chart(
    data=asia,
    metric='CO2_Mt',
    country='China',
    metric_name='CO₂ emissions',
    y_title='CO₂ emissions (million tonnes)',
    number_format=',.0f',
    unit=' Mt',
)
fig2.show()


Asian countries: China, India, Indonesia, Japan, South Korea


### Design choices

- Every country in each task is shown. Context lines use `#EEEEEE` at width 1.2.
- The selected country is drawn last, in teal at width 3.
- A direct end label identifies the selected country and its final value.
- The title calculates the actual percentage change between 2000 and 2022.
- Borders, gridlines, legends and rotated tick labels are removed.
- Task 2 uses the CSV’s `Region == "Asia"` classification and `CO2_Mt`.
